In [1]:
import numpy as np
import matplotlib.pyplot as plt
from pyprojroot import here

# ── Style matching make_figure_2.ipynb ──────────────────────────────────────
SMALL_SIZE  = 8
MEDIUM_SIZE = 9
BIGGER_SIZE = 12
TITLE_SIZE  = 9
LABEL_SIZE  = 8

plt.rc('font',   size=SMALL_SIZE)
plt.rc('axes',   titlesize=SMALL_SIZE)
plt.rc('axes',   labelsize=MEDIUM_SIZE)
plt.rc('xtick',  labelsize=SMALL_SIZE)
plt.rc('ytick',  labelsize=SMALL_SIZE)
plt.rc('legend', fontsize=SMALL_SIZE)
plt.rc('figure', titlesize=BIGGER_SIZE)

myblack = '#222222'

def customize_axes(ax):
    for spine in ax.spines.values():
        spine.set_color(myblack)
    ax.xaxis.label.set_color(myblack)
    ax.yaxis.label.set_color(myblack)
    ax.title.set_color(myblack)
    ax.tick_params(axis='x', colors=myblack)
    ax.tick_params(axis='y', colors=myblack)

In [2]:
HPC_DIR = here() / "05_bayesian_simulations" / "output"
FIG_DIR = here() / "figures"
FIG_DIR.mkdir(exist_ok=True)

# Fixed gamma value used throughout
GAMMA = 0.5


def load_and_average(N, m, delta, gamma=GAMMA, fs_str="0.6667", bayes=1):
    """
    Find all .npz files matching (N, m, delta, gamma), load mis_overall,
    and return the mean across files (averaging over different seeds).
    Prefers high-resolution (n51) files when multiple resolutions exist.
    """
    pattern = f"misperception_grid_N{N}_m{m}_fs{fs_str}_*_bayes{bayes}_delta{delta}_gamma{gamma}.npz"
    files = sorted(HPC_DIR.glob(pattern))

    if not files:
        raise FileNotFoundError(
            f"No files found for N={N}, m={m}, delta={delta}, gamma={gamma}\n"
            f"Pattern: {HPC_DIR / pattern}"
        )

    # Prefer n51 over coarser grids
    n51_files = [f for f in files if "_n51_" in f.name]
    chosen = n51_files if n51_files else files

    grids = []
    hss_vals = hoo_vals = None
    for f in chosen:
        data = np.load(f, allow_pickle=False)
        grids.append(data["mis_overall"])
        if hss_vals is None:
            hss_vals = data["hss_vals"]
            hoo_vals = data["hoo_vals"]

    mis_overall = np.mean(grids, axis=0)
    print(f"  N={N:4d}, m={m:2d}, δ={delta}, γ={gamma}: "
          f"{len(chosen)} file(s), grid {mis_overall.shape}")
    return hss_vals, hoo_vals, mis_overall


## Figure S1 — Overall misperception by network size
Rows: $N \in \{100, 500\}$, Columns: $\delta \in \{0.5, 1, 2\}$, fixed $m=5$, $\gamma=0.5$

In [3]:
Ns     = [100, 500]
deltas = [0.5, 1, 2]
m_fig1 = 5

print("Loading Figure S1 data (rows=N, cols=delta, m=5):")
data_fig1 = {}
for N in Ns:
    for delta in deltas:
        hss, hoo, mis = load_and_average(N, m_fig1, delta)
        data_fig1[(N, delta)] = (hss, hoo, mis)

# Shared symmetric colour scale across all panels
vmax = max(np.abs(d[2]).max() for d in data_fig1.values())
vmin = -vmax
print(f"\nShared colour scale: [{vmin:.3f}, {vmax:.3f}]")

Loading Figure S1 data (rows=N, cols=delta, m=5):
  N= 100, m= 5, δ=0.5, γ=0.5: 1 file(s), grid (51, 51)
  N= 100, m= 5, δ=1, γ=0.5: 1 file(s), grid (51, 51)
  N= 100, m= 5, δ=2, γ=0.5: 1 file(s), grid (51, 51)
  N= 500, m= 5, δ=0.5, γ=0.5: 1 file(s), grid (51, 51)
  N= 500, m= 5, δ=1, γ=0.5: 1 file(s), grid (51, 51)
  N= 500, m= 5, δ=2, γ=0.5: 1 file(s), grid (51, 51)

Shared colour scale: [-0.414, 0.414]


In [ ]:
tick_vals = np.linspace(0, 1, 5)
panel_letters = list("ABCDEF")
contour_levels = [0.1, 0.2, 0.25, 0.30]

fig, axes = plt.subplots(
    len(Ns), len(deltas),
    figsize=(7.5, 5.5),
    constrained_layout=False
)
plt.subplots_adjust(hspace=0.55, wspace=0.25)

im_last = None
for row_idx, N in enumerate(Ns):
    for col_idx, delta in enumerate(deltas):
        ax = axes[row_idx, col_idx]
        hss, hoo, mis = data_fig1[(N, delta)]
        letter = panel_letters[row_idx * len(deltas) + col_idx]

        im = ax.imshow(
            mis,
            extent=[0, 1, 0, 1],
            origin="lower",
            cmap="RdBu_r",
            vmin=vmin, vmax=vmax,
            aspect="equal"
        )
        im_last = im

        ax.set_xticks(tick_vals)
        ax.set_yticks(tick_vals)

        # Panel title: show both N and delta in every panel
        ax.set_title(
            rf"{letter}. $N={N},\ \delta={delta}$",
            color=myblack, fontsize=TITLE_SIZE
        )

        # y-label: left column only
        if col_idx == 0:
            ax.set_ylabel(r"$h_{oo}$", color=myblack, fontsize=LABEL_SIZE)
        else:
            ax.set_ylabel("")

        # x-label: bottom row only
        if row_idx == len(Ns) - 1:
            ax.set_xlabel(r"$h_{ss}$", color=myblack, fontsize=LABEL_SIZE)
        else:
            ax.set_xlabel("")

        customize_axes(ax)

        # Solid contours at non-zero levels
        X, Y = np.meshgrid(hss, hoo)
        cs = ax.contour(X, Y, mis, levels=contour_levels, colors=[myblack], linewidths=0.8)
        ax.clabel(cs, inline=True, fmt="%.2f", fontsize=7, colors=[myblack])

        # Zero contour (dotted, labeled)
        cs_zero = ax.contour(X, Y, mis, levels=[0], colors=[myblack], linewidths=0.8, linestyles='dotted')
        ax.clabel(cs_zero, inline=True, fmt="%.2f", fontsize=7, colors=[myblack])

# Suptitle
fig.suptitle(
    r"Overall misperception $\beta_{\mathrm{overall}}$ — varying $N$ and $\delta$ ($m=5$, $\gamma=0.5$)",
    color=myblack, fontsize=TITLE_SIZE, y=0.99
)

# Horizontal colorbar below panels (matching main-text figure style)
cbar_ax = fig.add_axes([0.15, -0.03, 0.70, 0.025])
cbar = fig.colorbar(im_last, cax=cbar_ax, orientation="horizontal")
cbar.set_label(
    r"Overall misperception $\beta_{\mathrm{overall}}$",
    color=myblack, fontsize=LABEL_SIZE
)
cbar.ax.xaxis.set_tick_params(color=myblack)
plt.setp(cbar.ax.get_xticklabels(), color=myblack)

out_path = FIG_DIR / "supp_bayesian_N_vs_delta.pdf"
plt.savefig(out_path, dpi=300, bbox_inches='tight')
print(f"Saved: {out_path}")
plt.show()

**Figure S1. Overall misperception $\beta_{\mathrm{overall}}$ as a function of homophily, for varying network sizes and Bayesian priors ($m=5$, $\gamma=0.5$).** Each panel shows a heatmap of overall misperception over the grid of supporter homophily $h_{ss}$ (x-axis) and opponent homophily $h_{oo}$ (y-axis), computed from preferential attachment networks with Bayesian rescaling. Columns vary the prior $\delta \in \{0.5, 1, 2\}$; rows vary the network size $N \in \{100, 500\}$. The number of edges added per new node is fixed at $m=5$ and the Bayesian uncertainty parameter at $\gamma=0.5$. Contour lines are drawn at $\beta_{\mathrm{overall}} \in \{0.10, 0.20, 0.25, 0.30\}$. Results are averaged across 500 Monte Carlo runs ($N=100$) or 100 runs ($N=500$) per grid cell.

## Figure S2 — Overall misperception by network density
Rows: $m \in \{2, 5, 10\}$, Columns: $\delta \in \{0.5, 1, 2\}$, fixed $N=500$, $\gamma=0.5$

In [5]:
ms     = [2, 5, 10]
deltas = [0.5, 1, 2]
N_fig2 = 500

print("Loading Figure S2 data (rows=m, cols=delta, N=500):")
data_fig2 = {}
for m in ms:
    for delta in deltas:
        hss, hoo, mis = load_and_average(N_fig2, m, delta)
        data_fig2[(m, delta)] = (hss, hoo, mis)

# Shared symmetric colour scale
vmax2 = max(np.abs(d[2]).max() for d in data_fig2.values())
vmin2 = -vmax2
print(f"\nShared colour scale: [{vmin2:.3f}, {vmax2:.3f}]")

Loading Figure S2 data (rows=m, cols=delta, N=500):
  N= 500, m= 2, δ=0.5, γ=0.5: 1 file(s), grid (51, 51)
  N= 500, m= 2, δ=1, γ=0.5: 1 file(s), grid (51, 51)
  N= 500, m= 2, δ=2, γ=0.5: 1 file(s), grid (51, 51)
  N= 500, m= 5, δ=0.5, γ=0.5: 1 file(s), grid (51, 51)
  N= 500, m= 5, δ=1, γ=0.5: 1 file(s), grid (51, 51)
  N= 500, m= 5, δ=2, γ=0.5: 1 file(s), grid (51, 51)
  N= 500, m=10, δ=0.5, γ=0.5: 1 file(s), grid (51, 51)
  N= 500, m=10, δ=1, γ=0.5: 1 file(s), grid (51, 51)
  N= 500, m=10, δ=2, γ=0.5: 1 file(s), grid (51, 51)

Shared colour scale: [-0.424, 0.424]


In [ ]:
panel_letters2 = list("ABCDEFGHI")
contour_levels = [0.1, 0.2, 0.25, 0.30]

fig2, axes2 = plt.subplots(
    len(ms), len(deltas),
    figsize=(7.5, 7.5),
    constrained_layout=False
)
plt.subplots_adjust(hspace=0.5, wspace=0.25)

im_last2 = None
for row_idx, m in enumerate(ms):
    for col_idx, delta in enumerate(deltas):
        ax = axes2[row_idx, col_idx]
        hss, hoo, mis = data_fig2[(m, delta)]
        letter = panel_letters2[row_idx * len(deltas) + col_idx]

        im = ax.imshow(
            mis,
            extent=[0, 1, 0, 1],
            origin="lower",
            cmap="RdBu_r",
            vmin=vmin2, vmax=vmax2,
            aspect="equal"
        )
        im_last2 = im

        ax.set_xticks(tick_vals)
        ax.set_yticks(tick_vals)

        # Panel title: show both m and delta in every panel
        ax.set_title(
            rf"{letter}. $m={m},\ \delta={delta}$",
            color=myblack, fontsize=TITLE_SIZE
        )

        # y-label: left column only
        if col_idx == 0:
            ax.set_ylabel(r"$h_{oo}$", color=myblack, fontsize=LABEL_SIZE)
        else:
            ax.set_ylabel("")

        # x-label: bottom row only
        if row_idx == len(ms) - 1:
            ax.set_xlabel(r"$h_{ss}$", color=myblack, fontsize=LABEL_SIZE)
        else:
            ax.set_xlabel("")

        customize_axes(ax)

        # Solid contours at non-zero levels
        X, Y = np.meshgrid(hss, hoo)
        cs = ax.contour(X, Y, mis, levels=contour_levels, colors=[myblack], linewidths=0.8)
        ax.clabel(cs, inline=True, fmt="%.2f", fontsize=7, colors=[myblack])

        # Zero contour (dotted, labeled)
        cs_zero = ax.contour(X, Y, mis, levels=[0], colors=[myblack], linewidths=0.8, linestyles='dotted')
        ax.clabel(cs_zero, inline=True, fmt="%.2f", fontsize=7, colors=[myblack])

# Suptitle
fig2.suptitle(
    r"Overall misperception $\beta_{\mathrm{overall}}$ — varying $m$ and $\delta$ ($N=500$, $\gamma=0.5$)",
    color=myblack, fontsize=TITLE_SIZE, y=0.95
)

# Horizontal colorbar below panels (matching main-text figure style)
cbar_ax2 = fig2.add_axes([0.15, -0.02, 0.70, 0.02])
cbar2 = fig2.colorbar(im_last2, cax=cbar_ax2, orientation="horizontal")
cbar2.set_label(
    r"Overall misperception $\beta_{\mathrm{overall}}$",
    color=myblack, fontsize=LABEL_SIZE
)
cbar2.ax.xaxis.set_tick_params(color=myblack)
plt.setp(cbar2.ax.get_xticklabels(), color=myblack)

out_path2 = FIG_DIR / "supp_bayesian_m_vs_delta.pdf"
plt.savefig(out_path2, dpi=300, bbox_inches='tight')
print(f"Saved: {out_path2}")
plt.show()

**Figure S2. Overall misperception $\beta_{\mathrm{overall}}$ as a function of homophily, for varying network densities and Bayesian priors ($N=500$, $\gamma=0.5$).** Each panel shows a heatmap of overall misperception over the grid of supporter homophily $h_{ss}$ (x-axis) and opponent homophily $h_{oo}$ (y-axis), computed from preferential attachment networks with Bayesian rescaling. Columns vary the prior $\delta \in \{0.5, 1, 2\}$; rows vary the number of edges added per new node $m \in \{2, 5, 10\}$. The network size is fixed at $N=500$ and the Bayesian uncertainty parameter at $\gamma=0.5$. Contour lines are drawn at $\beta_{\mathrm{overall}} \in \{0.10, 0.20, 0.25, 0.30\}$. Results are averaged across 100 Monte Carlo runs per grid cell.